# AI Handwriting Generator — Colab Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
- `HandwritingCNN`: CNN backbone + character embedding → 24 Bézier control points
- Input: real handwriting image (any sample from the target writer)
- Output: Bézier curves that reproduce that character in the writer's stroke style
- After training, download `checkpoints/style_net.pt` and place it in your local `checkpoints/` folder

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install opencv-contrib-python reportlab scikit-image scipy -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone / pull repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

os.chdir(REPO_PATH)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 5: Copy Writers_pngs from Drive
# Upload your Writers_pngs folder to: My Drive/Writers_pngs/
import shutil, os

src  = '/content/drive/MyDrive/Writers_pngs'
dest = f'{REPO_PATH}/Data/Writers_pngs'

if os.path.exists(dest) and len(os.listdir(dest)) > 2:
    print('Writers_pngs already present — skipping copy')
elif os.path.exists(src):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print('Writers_pngs copied successfully')
else:
    print('ERROR: Writers_pngs not found at', src)
    raise FileNotFoundError(src)

skip  = {'Writers_Zip', 'output_preview'}
total = 0
for d in sorted(os.listdir(dest)):
    if d in skip: continue
    p = os.path.join(dest, d)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'  {d}: {n} samples')
        total += n
print(f'Total: {total} samples')

In [ ]:
# Cell 6: Build Bézier label cache (skeleton extraction — runs once, ~5 min)
# This extracts Bézier control points from every real handwriting image.
# The cache is saved to Data/bezier_labels.npy so it only runs once.
import sys
sys.path.insert(0, f'{REPO_PATH}/src')

from data import load_all_data

DATA_ROOT  = f'{REPO_PATH}/Data/Writers_pngs'
CACHE_PATH = f'{REPO_PATH}/Data/bezier_labels.npy'

records, writer_names = load_all_data(DATA_ROOT, cache_path=CACHE_PATH)
print(f'\nWriters: {writer_names}')
print(f'Total records: {len(records)}')

In [ ]:
# Cell 7: Train the model
import sys
sys.path.insert(0, f'{REPO_PATH}/src')

import train as tr

# Override paths for Colab
tr.DATA_ROOT  = f'{REPO_PATH}/Data/Writers_pngs'
tr.CACHE_PATH = f'{REPO_PATH}/Data/bezier_labels.npy'
tr.CKPT_DIR   = f'{REPO_PATH}/checkpoints'
tr.NUM_EPOCHS = 100
tr.BATCH_SIZE = 64   # larger batch on GPU
tr.LR         = 1e-3

tr.main()

In [ ]:
# Cell 8: Save checkpoint to Drive + download locally
import shutil, os
from google.colab import files

ckpt_src  = f'{REPO_PATH}/checkpoints/style_net.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'style_net.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('Download started — put style_net.pt into your local checkpoints/ folder')

In [ ]:
# Cell 9: Quick generation demo in Colab
import sys, os
sys.path.insert(0, f'{REPO_PATH}/src')

from generate import load_model, generate_word
from PIL import Image
import matplotlib.pyplot as plt

model, ckpt, device, refs = load_model(f'{REPO_PATH}/checkpoints/style_net.pt')
print(f'Model loaded — epoch={ckpt["epoch"]}, val_MSE={ckpt["val_loss"]:.6f}')
print(f'Writers: {ckpt["writer_names"]}')

# Generate 'Hello' in writer 0's style
_, page = generate_word(model, 'Hello', writer_idx=0,
                        device=device, refs=refs, noise_scale=0.018)

plt.figure(figsize=(12, 3))
plt.imshow(page, cmap='gray')
plt.axis('off')
plt.title(f'Generated: Hello  (writer={ckpt["writer_names"][0]})')
plt.tight_layout()
plt.show()

# Try all writers
fig, axes = plt.subplots(len(refs), 1, figsize=(12, 3 * len(refs)))
if len(refs) == 1:
    axes = [axes]
for wid, ax in enumerate(axes):
    _, pg = generate_word(model, 'abc', writer_idx=wid,
                          device=device, refs=refs, noise_scale=0.018)
    ax.imshow(pg, cmap='gray')
    ax.axis('off')
    name = ckpt['writer_names'][wid] if wid < len(ckpt['writer_names']) else str(wid)
    ax.set_title(f'Writer {wid}: {name}')
plt.tight_layout()
plt.show()